# Week 5: Hospital General Information (Bronze, Silver, Gold join)

Purpose: add the second CMS dataset from the proposal (LO1), so H2 (quality), H3 (ownership) and the Week 6 regression can run.

Source: CMS Provider Data Catalog, *Hospital General Information* (dataset xubh-q36u), archived snapshot of 2024-10-30, file `Hospital_General_Information_2024-10-30.csv`. 5,394 hospitals x 39 columns. Join key: `Facility ID`, the 6-character CCN.

Outputs: `workspace.cms.bronze_hospital_info`, `workspace.cms.silver_hospital_info`, `workspace.cms.gold_dim_hospital_profile` (one row per inpatient hospital, joined to Gold on `ccn`).

Decisions (2026-11-02, revised 2026-11-03; logged in docs/data_quality_log.md)

1. Snapshot timing: use the CMS archived snapshot of 2024-10-30, inside the CY2024 claims year, so ratings, ownership and CCNs describe hospitals as they were when the claims were paid (DQ-24). The 2026 file was replaced on 2026-11-03 because 17 hospitals had changed CCN after converting to Critical Access, Rural Emergency or psychiatric status.

2. "Not Available" star ratings: kept as their own "Unrated" category. A numeric rating column (NULL for Unrated) is stored next to it so rated hospitals can still be ranked.

3. Ownership: 3 groups. For-profit = Proprietary + Physician. Non-profit = Voluntary non-profit (Private, Other, Church). Government = Local, Hospital District or Authority, State, Federal, Tribal. The original CMS value is kept in Silver.

4. Unmatched inpatient hospitals (7 expected): kept with NULL attributes and an `is_profile_matched` flag; excluded only from ownership and quality tests (DQ-25).

5. Gold design: a separate one-row-per-hospital table, `gold_dim_hospital_profile`, built here. `gold_dim_hospital` (notebook 03) is not modified, so every notebook stays rerunnable on its own.

In [0]:
# ---------------------------------------------------------------
# Cell 2: Config
# Purpose: every path, table name and expected value in one place
# Inputs: none | Outputs: Python variables used by later cells
# Expected values come from local profiling of the 2024-10-30 snapshot (2026-11-03)
# ---------------------------------------------------------------
CATALOG = "workspace"
SCHEMA = "cms"
VOLUME = "raw"
FILE_NAME = "Hospital_General_Information_2024-10-30.csv"

SOURCE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{FILE_NAME}"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_hospital_info"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_hospital_info"

EXPECTED_ROWS = 5_394
EXPECTED_SOURCE_COLS = 39

print("Source:", SOURCE_PATH)
print("Target:", BRONZE_TABLE)

Source: /Volumes/workspace/cms/raw/Hospital_General_Information_2024-10-30.csv
Target: workspace.cms.bronze_hospital_info


In [0]:
# ---------------------------------------------------------------
# Cell 3: Confirm the file landed in the volume
# Purpose: fail early if the CSV was not uploaded
# Inputs: /Volumes/workspace/cms/raw/ | Outputs: file listing
# Expected: 3 files (both Hospital General Information files and the inpatient CSV);
#           the 2024-10-30 file must be among them
# ---------------------------------------------------------------
files = dbutils.fs.ls(f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/")
display(files)
assert any(f.name == FILE_NAME for f in files), f"{FILE_NAME} not found in the volume"

path,name,size,modificationTime
dbfs:/Volumes/workspace/cms/raw/Hospital_General_Information.csv,Hospital_General_Information.csv,1450767,1793687157000
dbfs:/Volumes/workspace/cms/raw/Hospital_General_Information_2024-10-30.csv,Hospital_General_Information_2024-10-30.csv,1468150,1793740987000
dbfs:/Volumes/workspace/cms/raw/MUP_INP_RY26_P03_V10_DY24_PrvSvc.CSV,MUP_INP_RY26_P03_V10_DY24_PrvSvc.CSV,37976130,1791494088000


In [0]:
# ---------------------------------------------------------------
# Cell 4: Bronze ingest
# Purpose: land the CMS file as-is, every column STRING, plus lineage
# Inputs: SOURCE_PATH | Outputs: workspace.cms.bronze_hospital_info
# Expected: 5,394 rows, 39 source columns, 0 corrupt records
# Same pattern as 01_bronze_ingest.py
# ---------------------------------------------------------------
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

# Step A: read only the header row to get the column names exactly as CMS wrote them
header_cols = spark.read.option("header", True).csv(SOURCE_PATH).columns
assert len(header_cols) == EXPECTED_SOURCE_COLS, f"Expected {EXPECTED_SOURCE_COLS} columns, got {len(header_cols)}"

# Step B: explicit all-string schema, so Facility ID keeps its leading zeros
# and any malformed line lands in _corrupt_record instead of being dropped
bronze_schema = StructType(
    [StructField(c, StringType(), True) for c in header_cols]
    + [StructField("_corrupt_record", StringType(), True)]
)

raw_df = (
    spark.read
    .schema(bronze_schema)
    .option("header", True)
    .option("enforceSchema", False)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", False)
    .option("encoding", "UTF-8")
    .csv(SOURCE_PATH)
)

# Step C: add lineage columns, the only things Bronze adds
bronze_df = (
    raw_df
    .withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

# Write with Delta column mapping so the CMS column names (spaces, /) are kept as-is
bronze_df.createOrReplaceTempView("bronze_hospital_info_src")
spark.sql(f"CREATE OR REPLACE TABLE {BRONZE_TABLE} TBLPROPERTIES ('delta.columnMapping.mode' = 'name') AS SELECT * FROM bronze_hospital_info_src")

spark.sql(f"""
COMMENT ON TABLE {BRONZE_TABLE} IS
'Bronze: CMS Hospital General Information (Provider Data Catalog xubh-q36u, snapshot 2024-10-30, archived), loaded as-is. All source columns are STRING. One row = one hospital (Facility ID = CCN). Lineage: _ingest_ts, _source_file.'
""")

# Step D: row count and corrupt records
bronze = spark.table(BRONZE_TABLE)
n_rows = bronze.count()
n_corrupt = bronze.filter(F.col("_corrupt_record").isNotNull()).count()
print("Rows:", n_rows, "| Corrupt records:", n_corrupt)
assert n_rows == EXPECTED_ROWS, f"Expected {EXPECTED_ROWS} rows, got {n_rows}"
assert n_corrupt == 0, f"{n_corrupt} corrupt records"
print("Wrote", BRONZE_TABLE)

Rows: 5394 | Corrupt records: 0
Wrote workspace.cms.bronze_hospital_info


In [0]:
%sql
--- ---------------------------------------------------------------
--- Cell 5: Bronze checks
--- Purpose: confirm the key column is complete, unique and 6 characters
--- Inputs: workspace.cms.bronze_hospital_info | Outputs: one result row
--- Expected: 5,394 rows; 5,394 distinct Facility IDs; 0 NULL IDs; 0 IDs not 6 characters
--- ---------------------------------------------------------------

SELECT 
    COUNT(*) AS n_rows,
    COUNT (DISTINCT `Facility ID`) AS n_distinct_ids,
    SUM(CASE WHEN `Facility ID` IS NULL THEN 1 ELSE 0 END) AS n_null_ids,
    SUM(CASE WHEN LENGTH(`Facility ID`) != 6 THEN 1 ELSE 0 END) AS n_not_6_chars

FROM    
    workspace.cms.bronze_hospital_info;


n_rows,n_distinct_ids,n_null_ids,n_not_6_chars
5394,5394,0,0


In [0]:
%sql
--- ---------------------------------------------------------------
--- Cell 6a: Hospital Type profile
--- Purpose: see which hospital types exist; the inpatient file should match only one
--- Inputs: workspace.cms.bronze_hospital_info | Outputs: one row per type
--- Expected: 6 types; Acute Care Hospitals 3,153 is the largest
--- ---------------------------------------------------------------

SELECT `Hospital Type`, COUNT(*) AS n_hospitals
FROM workspace.cms.bronze_hospital_info
GROUP BY `Hospital Type`
ORDER BY n_hospitals DESC;


Hospital Type,n_hospitals
Acute Care Hospitals,3153
Critical Access Hospitals,1370
Psychiatric,615
Acute Care - Veterans Administration,132
Childrens,92
Acute Care - Department of Defense,32


In [0]:
%sql
--- ---------------------------------------------------------------
--- Cell 6b: Hospital Ownership profile
--- Purpose: every value must map to an ownership group in Silver (decision 3)
--- Inputs: workspace.cms.bronze_hospital_info | Outputs: one row per ownership value
--- Expected: 12 values; Voluntary non-profit - Private 2,277 is the largest
--- ---------------------------------------------------------------

SELECT `Hospital Ownership`, COUNT(*) AS n_hospitals
FROM workspace.cms.bronze_hospital_info
GROUP BY `Hospital Ownership`
ORDER BY n_hospitals DESC;


Hospital Ownership,n_hospitals
Voluntary non-profit - Private,2277
Proprietary,1050
Government - Hospital District or Authority,514
Government - Local,401
Voluntary non-profit - Other,369
Voluntary non-profit - Church,280
Government - State,206
Veterans Health Administration,132
Physician,76
Government - Federal,42


In [0]:
%sql
--- ---------------------------------------------------------------
--- Cell 6c: Hospital overall rating profile
--- Purpose: confirm the rating values before typing them in Silver (decision 2)
--- Inputs: workspace.cms.bronze_hospital_info | Outputs: one row per rating value
--- Expected: 6 values: 1 to 5 plus "Not Available" (2,568)
--- ---------------------------------------------------------------

SELECT `Hospital overall rating`, COUNT(*) AS n_hospitals
FROM workspace.cms.bronze_hospital_info
GROUP BY `Hospital overall rating`
ORDER BY `Hospital overall rating`;


Hospital overall rating,n_hospitals
1,276
2,588
3,822
4,759
5,381
Not Available,2568


In [0]:
%sql
--- ---------------------------------------------------------------
--- Cell 7: Join preview
--- Purpose: how many inpatient hospitals find a match before Silver is built
--- Inputs: workspace.cms.gold_dim_hospital, workspace.cms.bronze_hospital_info | Outputs: counts
--- Expected: 2,906 inpatient hospitals; 2,899 matched; 7 unmatched
--- Silver ccn is already 6 digits (DQ-02), so no LPAD is needed
--- ---------------------------------------------------------------

SELECT 
    COUNT(*) AS n_inpatient_hospitals,
    COUNT (b.`Facility ID`) AS n_matched,
    COUNT (*) - COUNT (b.`Facility ID`) AS n_unmatched

FROM workspace.cms.gold_dim_hospital h 
LEFT JOIN workspace.cms.bronze_hospital_info b 
    ON h.ccn = b.`Facility ID`;


n_inpatient_hospitals,n_matched,n_unmatched
2906,2899,7


In [0]:
%sql
--- ---------------------------------------------------------------
--- Cell 7b: Unmatched inpatient hospitals
--- Purpose: list the hospitals with no Hospital General Information row, to explain them in DQ-25
--- Inputs: workspace.cms.gold_dim_hospital, workspace.cms.bronze_hospital_info | Outputs: one row per unmatched hospital
--- Expected: 7 rows
--- ---------------------------------------------------------------

SELECT 
    h.ccn, 
    h.provider_name, 
    h.provider_city, 
    h.state_abbr

FROM 
    workspace.cms.gold_dim_hospital h

LEFT JOIN 
    workspace.cms.bronze_hospital_info b 
    ON h.ccn = b.`Facility ID`

WHERE
    b.`Facility ID` IS NULL 

ORDER BY 
    h.state_abbr, h.provider_name;


ccn,provider_name,provider_city,state_abbr
100364,Adventhealth Riverview,Riverview,FL
140310,"Rush Specialty Hospital, Llc",Chicago,IL
170208,"Adventhealth South Overland Park, Inc",Overland Park,KS
210006,University Of Md Harford Memorial Hospital,Havre De Grace,MD
360374,Mercy Health Kings Mills Hospital Llc,Mason,OH
670333,Westover Hills Baptist Hospital,San Antonio,TX
520013,Sacred Heart Hospital,Eau Claire,WI


## Silver

Next: type the rating (numeric column, NULL for Unrated, plus a rating category with "Unrated"), map ownership to the three groups (plus Federal (VA/DoD) for the two federal values that have no inpatient claims), keep the original CMS columns for traceability, and add checks.

In [0]:
%sql
--- ---------------------------------------------------------------
--- Cell 8: Silver hospital attributes
--- Purpose: type and clean the columns the analysis uses; map ownership to groups (decisions 2 and 3)
--- Inputs: workspace.cms.bronze_hospital_info | Outputs: workspace.cms.silver_hospital_info
--- Expected: 5,394 rows (no rows removed; the join to inpatient happens in Gold)
--- ---------------------------------------------------------------

CREATE OR REPLACE TABLE workspace.cms.silver_hospital_info AS

SELECT 
    `Facility ID` AS ccn,
    `Facility Name` AS facility_name,
    `State` AS state_abbr,
    `Hospital Type` AS hospital_type,
    `Hospital Ownership` AS hospital_ownership,

    CASE 
        WHEN `Hospital Ownership` IN ('Proprietary', 'Physician') THEN 'For-profit'
        WHEN `Hospital Ownership` LIKE 'Voluntary non-profit%' THEN 'Non-profit'
        WHEN `Hospital Ownership` LIKE 'Government%' OR `Hospital Ownership` = 'Tribal' THEN 'Government'
        WHEN `Hospital Ownership` IN ('Department of Defense', 'Veterans Health Administration') THEN 'Federal (VA/DoD)'

    END AS ownership_group,

    `Emergency Services` = 'Yes' AS has_emergency_services, 

    CASE WHEN `Hospital overall rating` = 'Not Available' THEN NULL 
    ELSE CAST(`Hospital overall rating` AS INT) END AS star_rating,

    CASE WHEN `Hospital overall rating` = 'Not Available' THEN 'Unrated'
        ELSE `Hospital overall rating` END AS rating_category

FROM 
    workspace.cms.bronze_hospital_info;

num_affected_rows,num_inserted_rows


In [0]:
%sql
--- ---------------------------------------------------------------
--- Cell 9a: Silver row and key checks
--- Purpose: confirm Silver kept every row and the key is still unique
--- Inputs: workspace.cms.silver_hospital_info | Outputs: one result row
--- Expected: 5,394 rows; 5,394 distinct ccn; 0 NULL ownership_group (every value mapped)
--- ---------------------------------------------------------------

SELECT COUNT(*) AS  n_rows, 
    COUNT(DISTINCT ccn) AS n_distinct_ccn, 
    SUM(CASE WHEN ownership_group IS NULL THEN 1 ELSE 0 END) AS n_null_ownership_group
    
FROM workspace.cms.silver_hospital_info

n_rows,n_distinct_ccn,n_null_ownership_group
5394,5394,0


In [0]:
%sql
--- ---------------------------------------------------------------
--- Cell 9b: Ownership group counts
--- Purpose: confirm each CMS value landed in the right group (decision 3)
--- Inputs: workspace.cms.silver_hospital_info | Outputs: one row per group
--- Expected: Non-profit 2,926; Government 1,178; For-profit 1,126; Federal (VA/DoD) 164 (total 5,394)
--- ---------------------------------------------------------------

SELECT ownership_group, 
    COUNT(*) AS n_hospitals 
FROM workspace.cms.silver_hospital_info 
GROUP BY ownership_group
ORDER BY ownership_group;

ownership_group,n_hospitals
Federal (VA/DoD),164
For-profit,1126
Government,1178
Non-profit,2926


In [0]:
%sql
--- ---------------------------------------------------------------
--- Cell 9c: Rating checks
--- Purpose: confirm star_rating and rating_category agree (decision 2)
--- Inputs: workspace.cms.silver_hospital_info | Outputs: one row per rating_category
--- Expected: 6 categories; Unrated 2,568 with star_rating NULL; '1' to '5' each with the same number
---           as star_rating (1: 276, 2: 588, 3: 822, 4: 759, 5: 381)
--- ---------------------------------------------------------------

SELECT rating_category, 
    COUNT(*) AS n_hospitals, 
    COUNT(*) = SUM(CASE WHEN star_rating IS NULL THEN 0 ELSE 1 END) AS all_rows_have_star_rating,
    MIN(star_rating),
    MAX(star_rating) 
FROM workspace.cms.silver_hospital_info 
GROUP BY rating_category
ORDER BY rating_category;

rating_category,n_hospitals,all_rows_have_star_rating,MIN(star_rating),MAX(star_rating)
1,276,true,1,1
2,588,true,2,2
3,822,true,3,3
4,759,true,4,4
5,381,true,5,5
Unrated,2568,false,null,null


In [0]:
%sql
--- ---------------------------------------------------------------
--- Cell 10: Gold hospital profile
--- Purpose: one row per inpatient hospital with its ownership and quality attributes (decisions 4 and 5)
--- Inputs: workspace.cms.gold_dim_hospital, workspace.cms.silver_hospital_info
--- Outputs: workspace.cms.gold_dim_hospital_profile
--- Expected: 2,906 rows (every inpatient hospital, none dropped, none duplicated)
--- ---------------------------------------------------------------

CREATE OR REPLACE TABLE workspace.cms.gold_dim_hospital_profile
AS
SELECT h.ccn, 
    si.hospital_type, 
    si.hospital_ownership, 
    si.ownership_group, 
    si.has_emergency_services, 
    si.star_rating, 
    si.rating_category, 
    CASE WHEN si.ccn IS NULL THEN false ELSE true END AS is_profile_matched
FROM workspace.cms.gold_dim_hospital h
LEFT JOIN workspace.cms.silver_hospital_info si
ON h.ccn = si.ccn

num_affected_rows,num_inserted_rows


In [0]:
%sql
--- ---------------------------------------------------------------
--- Cell 11a: Gold profile row, key and match checks
--- Purpose: confirm every inpatient hospital is kept once, and the match flag agrees with Cell 7
--- Inputs: workspace.cms.gold_dim_hospital_profile | Outputs: one result row
--- Expected: 2,906 rows; 2,906 distinct ccn; 2,899 matched; 7 unmatched
--- ---------------------------------------------------------------

SELECT COUNT(*) AS  n_rows, 
    COUNT(DISTINCT ccn) AS n_distinct_ccn, 
    SUM(CASE WHEN is_profile_matched THEN 1 ELSE 0 END) AS n_matched, 
    SUM(CASE WHEN NOT is_profile_matched THEN 1 ELSE 0 END) AS n_unmatched
FROM workspace.cms.gold_dim_hospital_profile;

n_rows,n_distinct_ccn,n_matched,n_unmatched
2906,2906,2899,7


In [0]:
%sql
--- ---------------------------------------------------------------
--- Cell 11b: Ownership and type among inpatient hospitals
--- Purpose: see the ownership mix the H3 test and the regression will use
--- Inputs: workspace.cms.gold_dim_hospital_profile | Outputs: one row per ownership group
--- Expected: Non-profit 1,875; For-profit 631; Government 393; NULL 7 (unmatched); no Federal (VA/DoD)
---           Every matched hospital is 'Acute Care Hospitals'
--- ---------------------------------------------------------------

SELECT ownership_group, 
    COUNT(*) AS n_hospitals, 
    COUNT(DISTINCT hospital_type) AS n_distinct_hospital_types
FROM workspace.cms.gold_dim_hospital_profile
GROUP BY ownership_group
ORDER BY ownership_group;

ownership_group,n_hospitals,n_distinct_hospital_types
null,7,0
For-profit,631,1
Government,393,1
Non-profit,1875,1


In [0]:
%sql
--- ---------------------------------------------------------------
--- Cell 11c: Star ratings among inpatient hospitals
--- Purpose: see the rating mix the H2 test and the regression will use (decision 2)
--- Inputs: workspace.cms.gold_dim_hospital_profile | Outputs: one row per rating_category
--- Expected: 1: 250; 2: 529; 3: 750; 4: 677; 5: 311; Unrated 382; NULL 7 (unmatched)
---           star_rating min and max equal the category for 1 to 5; NULL for Unrated and unmatched
--- ---------------------------------------------------------------

SELECT rating_category, 
    COUNT(*) AS n_hospitals, 
    MIN(star_rating) AS min_star_rating, 
    MAX(star_rating) AS max_star_rating
FROM workspace.cms.gold_dim_hospital_profile
GROUP BY rating_category
ORDER BY rating_category;

rating_category,n_hospitals,min_star_rating,max_star_rating
null,7,null,null
1,250,1,1
2,529,2,2
3,750,3,3
4,677,4,4
5,311,5,5
Unrated,382,null,null
